# 🏗️ Enterprise Infrastructure Provisioning — IaC for Fabric

**Comprehensive infrastructure-as-code deployment for Microsoft Fabric workspaces and artifacts**

## Coverage

### ✅ Infrastructure-as-Code Tools
- **Terraform** — Multi-cloud IaC standard (recommended for enterprise)
- **Bicep** — Azure-native declarative language
- **Fabric CLI** — Fabric REST API automation

### 📦 Provisioned Resources

**Workspaces:**
- Premium capacity assignment (F SKU)
- RBAC assignments (admin, member, contributor, viewer)
- Git integration configuration
- Settings and governance policies

**Data Artifacts:**
- Lakehouses (Bronze, Silver, Gold, Metadata)
- Warehouses (SQL analytics)
- Notebooks (all 17 production notebooks)
- Data pipelines
- Semantic models (Power BI datasets)
- Reports and dashboards

**AI/ML Artifacts:**
- ML models
- Experiments
- Environments (Spark runtime config)

**Real-Time Artifacts:**
- KQL databases
- Eventstreams
- Eventhouse

## Enterprise Capabilities

✅ **Multi-environment** — Dev, Test, Prod with environment-specific configs  
✅ **Idempotent** — Safe to re-run without duplication  
✅ **Modular** — Reusable components  
✅ **Parameterized** — Config-driven deployment  
✅ **CI/CD ready** — GitHub Actions, Azure DevOps integration  
✅ **State management** — Terraform state, deployment tracking  
✅ **RBAC enforcement** — Least-privilege access  

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Load Fabric Utilities                                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
%run 00_fabric_native_utils

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: Infrastructure Configuration Schema                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from typing import Dict, List, Optional
from datetime import datetime

print("🏗️ Enterprise Infrastructure Provisioning")
print("=" * 80)

# === Infrastructure Configuration Schema ===

infrastructure_config_schema = {
    "environments": {
        "dev": {
            "capacity_id": "00000000-0000-0000-0000-000000000000",
            "capacity_sku": "F2",
            "region": "eastus",
            "workspaces": [
                {
                    "name": "insurance-dev",
                    "description": "Development environment for Insurance Accelerator",
                    "rbac": [
                        {"principal": "dev-team@company.com", "role": "Admin"},
                        {"principal": "data-engineers@company.com", "role": "Member"}
                    ],
                    "git_integration": {
                        "enabled": True,
                        "repository": "https://github.com/deepugun/insurance-fabric-accelerator",
                        "branch": "develop",
                        "directory": "/notebooks"
                    },
                    "artifacts": {
                        "lakehouses": [
                            {"name": "insurance_bronze_dev", "description": "Raw data - Dev"},
                            {"name": "insurance_silver_dev", "description": "Cleansed data - Dev"},
                            {"name": "insurance_gold_dev", "description": "Business-ready data - Dev"},
                            {"name": "insurance_metadata_dev", "description": "Operational metadata - Dev"}
                        ],
                        "warehouses": [
                            {"name": "insurance_analytics_dev", "description": "SQL analytics - Dev"}
                        ],
                        "notebooks": [
                            "00_fabric_native_utils.ipynb",
                            "01_demo_data_generator.ipynb",
                            "10_admin_governance_console.ipynb",
                            "15_fabric_mirroring_setup.ipynb",
                            "20_ai_showcase_all_features.ipynb",
                            "25_document_extraction_foundry_summarization.ipynb",
                            "30_medallion_transformations.ipynb",
                            "35_streaming_realtime_intelligence.ipynb",
                            "40_data_quality_framework.ipynb",
                            "45_operational_monitoring.ipynb",
                            "50_security_compliance.ipynb",
                            "55_external_data_exchange_sftp.ipynb",
                            "56_access_monitoring_control.ipynb",
                            "60_test_runner.ipynb",
                            "70_cicd_deployment_automation.ipynb",
                            "80_fabric_iq_ontology.ipynb",
                            "90_central_cockpit_dashboard.ipynb",
                            "99_marketplace_deployment.ipynb"
                        ],
                        "environments": [
                            {
                                "name": "insurance-python-env-dev",
                                "description": "Python packages for Insurance workloads",
                                "libraries": ["pandas", "numpy", "scikit-learn", "xgboost", "lightgbm"]
                            }
                        ],
                        "kql_databases": [
                            {"name": "insurance_realtime_dev", "description": "Real-time analytics - Dev"}
                        ]
                    }
                }
            ]
        },
        "test": {
            "capacity_id": "11111111-1111-1111-1111-111111111111",
            "capacity_sku": "F4",
            "region": "eastus",
            "workspaces": [
                {
                    "name": "insurance-test",
                    "description": "Testing environment for Insurance Accelerator",
                    "rbac": [
                        {"principal": "qa-team@company.com", "role": "Admin"},
                        {"principal": "data-engineers@company.com", "role": "Member"}
                    ],
                    "artifacts": {
                        "lakehouses": [
                            {"name": "insurance_bronze_test", "description": "Raw data - Test"},
                            {"name": "insurance_silver_test", "description": "Cleansed data - Test"},
                            {"name": "insurance_gold_test", "description": "Business-ready data - Test"},
                            {"name": "insurance_metadata_test", "description": "Operational metadata - Test"}
                        ],
                        "warehouses": [
                            {"name": "insurance_analytics_test", "description": "SQL analytics - Test"}
                        ]
                    }
                }
            ]
        },
        "prod": {
            "capacity_id": "22222222-2222-2222-2222-222222222222",
            "capacity_sku": "F64",
            "region": "eastus2",
            "workspaces": [
                {
                    "name": "insurance-prod",
                    "description": "Production environment for Insurance Accelerator",
                    "rbac": [
                        {"principal": "prod-admins@company.com", "role": "Admin"},
                        {"principal": "data-engineers@company.com", "role": "Contributor"},
                        {"principal": "business-users@company.com", "role": "Viewer"}
                    ],
                    "git_integration": {
                        "enabled": True,
                        "repository": "https://github.com/deepugun/insurance-fabric-accelerator",
                        "branch": "main",
                        "directory": "/notebooks"
                    },
                    "artifacts": {
                        "lakehouses": [
                            {"name": "insurance_bronze", "description": "Raw data - Production"},
                            {"name": "insurance_silver", "description": "Cleansed data - Production"},
                            {"name": "insurance_gold", "description": "Business-ready data - Production"},
                            {"name": "insurance_metadata", "description": "Operational metadata - Production"}
                        ],
                        "warehouses": [
                            {"name": "insurance_analytics", "description": "SQL analytics - Production"}
                        ],
                        "kql_databases": [
                            {"name": "insurance_realtime", "description": "Real-time analytics - Production"}
                        ],
                        "mirrored_databases": [
                            {"name": "PolicySystem_Mirror", "description": "Policy Admin CDC mirror"},
                            {"name": "ClaimsSystem_Mirror", "description": "Claims Management CDC mirror"},
                            {"name": "FinanceSystem_Mirror", "description": "Finance System CDC mirror"},
                            {"name": "CRM_Mirror", "description": "CRM CDC mirror"}
                        ]
                    }
                }
            ]
        }
    }
}

# Save configuration to Delta table for reference
config_df = spark.createDataFrame([{
    "config_version": "1.0",
    "config_json": json.dumps(infrastructure_config_schema, indent=2),
    "created_timestamp": datetime.now()
}])

config_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata.infrastructure_config")

print("\n✅ Infrastructure configuration schema loaded")
print(f"   Environments: {', '.join(infrastructure_config_schema['environments'].keys())}")
print("=" * 80)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: Terraform Configuration Generator                       ║
# ╚══════════════════════════════════════════════════════════════════════╝

class TerraformGenerator:
    """
    Generate Terraform configurations for Microsoft Fabric.
    Uses Fabric REST API provider.
    """
    
    @staticmethod
    def generate_provider_config() -> str:
        """Generate Terraform provider configuration."""
        return '''terraform {
  required_version = ">= 1.6"
  
  required_providers {
    azurerm = {
      source  = "hashicorp/azurerm"
      version = "~> 3.0"
    }
    azapi = {
      source  = "Azure/azapi"
      version = "~> 1.0"
    }
  }
  
  backend "azurerm" {
    resource_group_name  = "insurance-tfstate-rg"
    storage_account_name = "insurancetfstate"
    container_name       = "tfstate"
    key                  = "fabric.terraform.tfstate"
  }
}

provider "azurerm" {
  features {}
}

provider "azapi" {
  # Uses Azure CLI or Managed Identity authentication
}
'''
    
    @staticmethod
    def generate_variables(environment: str) -> str:
        """Generate Terraform variables for environment."""
        return f'''variable "environment" {{
  description = "Environment name (dev, test, prod)"
  type        = string
  default     = "{environment}"
}}

variable "location" {{
  description = "Azure region"
  type        = string
  default     = "eastus"
}}

variable "capacity_sku" {{
  description = "Fabric capacity SKU"
  type        = string
  default     = "F64"
}}

variable "resource_group_name" {{
  description = "Resource group for Fabric capacity"
  type        = string
  default     = "insurance-fabric-rg"
}}

variable "capacity_admins" {{
  description = "List of capacity admin email addresses"
  type        = list(string)
  default     = ["admin@company.com"]
}}

variable "tags" {{
  description = "Tags for all resources"
  type        = map(string)
  default     = {{
    Project     = "Insurance Accelerator"
    Environment = "{environment}"
    ManagedBy   = "Terraform"
    CostCenter  = "DataPlatform"
  }}
}}
'''
    
    @staticmethod
    def generate_capacity(env_config: Dict) -> str:
        """Generate Terraform for Fabric capacity."""
        sku = env_config.get("capacity_sku", "F64")
        region = env_config.get("region", "eastus")
        
        return f'''# Fabric Capacity
resource "azapi_resource" "fabric_capacity" {{
  type      = "Microsoft.Fabric/capacities@2023-11-01"
  name      = "insurance-fabric-${{var.environment}}"
  location  = var.location
  parent_id = azurerm_resource_group.fabric_rg.id

  body = jsonencode({{
    sku = {{
      name = "{sku}"
      tier = "Fabric"
    }}
    properties = {{
      administration = {{
        members = var.capacity_admins
      }}
    }}
  }})

  tags = var.tags
}}

# Output capacity ID for workspace assignment
output "capacity_id" {{
  value = azapi_resource.fabric_capacity.id
}}
'''
    
    @staticmethod
    def generate_workspace(workspace_config: Dict, capacity_id: str) -> str:
        """Generate Terraform for Fabric workspace."""
        name = workspace_config["name"]
        description = workspace_config.get("description", "")
        
        tf = f'''# Workspace: {name}
resource "azapi_resource" "workspace_{name.replace("-", "_")}" {{
  type      = "Microsoft.Fabric/workspaces@2023-11-01"
  name      = "{name}"
  parent_id = azapi_resource.fabric_capacity.id

  body = jsonencode({{
    properties = {{
      displayName = "{name}"
      description = "{description}"
      capacityId  = azapi_resource.fabric_capacity.id
    }}
  }})

  tags = var.tags
}}
'''
        
        # Add RBAC assignments
        rbac = workspace_config.get("rbac", [])
        for i, assignment in enumerate(rbac):
            principal = assignment["principal"]
            role = assignment["role"]
            
            tf += f'''
# RBAC: {principal} as {role}
resource "azapi_resource" "workspace_{name.replace("-", "_")}_rbac_{i}" {{
  type      = "Microsoft.Fabric/workspaces/roleAssignments@2023-11-01"
  name      = "rbac-{i}"
  parent_id = azapi_resource.workspace_{name.replace("-", "_")}.id

  body = jsonencode({{
    properties = {{
      principal = "{principal}"
      role      = "{role}"
    }}
  }})
}}
'''
        
        return tf
    
    @staticmethod
    def generate_lakehouse(lakehouse_config: Dict, workspace_name: str) -> str:
        """Generate Terraform for lakehouse."""
        name = lakehouse_config["name"]
        description = lakehouse_config.get("description", "")
        workspace_tf_name = workspace_name.replace("-", "_")
        
        return f'''# Lakehouse: {name}
resource "azapi_resource" "lakehouse_{name}" {{
  type      = "Microsoft.Fabric/workspaces/lakehouses@2023-11-01"
  name      = "{name}"
  parent_id = azapi_resource.workspace_{workspace_tf_name}.id

  body = jsonencode({{
    properties = {{
      displayName = "{name}"
      description = "{description}"
    }}
  }})
}}
'''
    
    @staticmethod
    def generate_warehouse(warehouse_config: Dict, workspace_name: str) -> str:
        """Generate Terraform for warehouse."""
        name = warehouse_config["name"]
        description = warehouse_config.get("description", "")
        workspace_tf_name = workspace_name.replace("-", "_")
        
        return f'''# Warehouse: {name}
resource "azapi_resource" "warehouse_{name}" {{
  type      = "Microsoft.Fabric/workspaces/warehouses@2023-11-01"
  name      = "{name}"
  parent_id = azapi_resource.workspace_{workspace_tf_name}.id

  body = jsonencode({{
    properties = {{
      displayName = "{name}"
      description = "{description}"
    }}
  }})
}}
'''
    
    @staticmethod
    def generate_complete_terraform(environment: str, config: Dict) -> str:
        """Generate complete Terraform configuration for environment."""
        
        env_config = config["environments"][environment]
        
        tf_config = TerraformGenerator.generate_provider_config()
        tf_config += "\n\n"
        tf_config += TerraformGenerator.generate_variables(environment)
        tf_config += "\n\n"
        
        # Resource group
        tf_config += '''# Resource Group
resource "azurerm_resource_group" "fabric_rg" {
  name     = var.resource_group_name
  location = var.location
  tags     = var.tags
}

'''
        
        # Capacity
        tf_config += TerraformGenerator.generate_capacity(env_config)
        tf_config += "\n\n"
        
        # Workspaces and artifacts
        for workspace_config in env_config["workspaces"]:
            workspace_name = workspace_config["name"]
            
            # Workspace
            tf_config += TerraformGenerator.generate_workspace(
                workspace_config, 
                env_config.get("capacity_id", "")
            )
            tf_config += "\n\n"
            
            # Lakehouses
            for lakehouse in workspace_config.get("artifacts", {}).get("lakehouses", []):
                tf_config += TerraformGenerator.generate_lakehouse(lakehouse, workspace_name)
                tf_config += "\n\n"
            
            # Warehouses
            for warehouse in workspace_config.get("artifacts", {}).get("warehouses", []):
                tf_config += TerraformGenerator.generate_warehouse(warehouse, workspace_name)
                tf_config += "\n\n"
        
        return tf_config

print("\n✅ TerraformGenerator class loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: Bicep Configuration Generator                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

class BicepGenerator:
    """
    Generate Bicep configurations for Microsoft Fabric.
    Azure-native IaC language.
    """
    
    @staticmethod
    def generate_parameters(environment: str) -> str:
        """Generate Bicep parameters file."""
        return f'''{{
  "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
  "contentVersion": "1.0.0.0",
  "parameters": {{
    "environment": {{
      "value": "{environment}"
    }},
    "location": {{
      "value": "eastus"
    }},
    "capacitySku": {{
      "value": "F64"
    }},
    "tags": {{
      "value": {{
        "Project": "Insurance Accelerator",
        "Environment": "{environment}",
        "ManagedBy": "Bicep",
        "CostCenter": "DataPlatform"
      }}
    }}
  }}
}}
'''
    
    @staticmethod
    def generate_capacity_bicep(env_config: Dict) -> str:
        """Generate Bicep for Fabric capacity."""
        sku = env_config.get("capacity_sku", "F64")
        
        return f'''// Fabric Capacity
param environment string
param location string = resourceGroup().location
param capacitySku string = '{sku}'
param tags object = {{}}

resource fabricCapacity 'Microsoft.Fabric/capacities@2023-11-01' = {{
  name: 'insurance-fabric-${{environment}}'
  location: location
  sku: {{
    name: capacitySku
    tier: 'Fabric'
  }}
  properties: {{
    administration: {{
      members: [
        'admin@company.com'
      ]
    }}
  }}
  tags: tags
}}

output capacityId string = fabricCapacity.id
output capacityName string = fabricCapacity.name
'''
    
    @staticmethod
    def generate_workspace_bicep(workspace_config: Dict) -> str:
        """Generate Bicep for workspace and artifacts."""
        name = workspace_config["name"]
        description = workspace_config.get("description", "")
        
        bicep = f'''// Workspace: {name}
param capacityId string

resource workspace 'Microsoft.Fabric/workspaces@2023-11-01' = {{
  name: '{name}'
  properties: {{
    displayName: '{name}'
    description: '{description}'
    capacityId: capacityId
  }}
}}
'''
        
        # Add lakehouses
        lakehouses = workspace_config.get("artifacts", {}).get("lakehouses", [])
        for lakehouse in lakehouses:
            lakehouse_name = lakehouse["name"]
            lakehouse_desc = lakehouse.get("description", "")
            
            bicep += f'''
resource lakehouse_{lakehouse_name} 'Microsoft.Fabric/workspaces/lakehouses@2023-11-01' = {{
  parent: workspace
  name: '{lakehouse_name}'
  properties: {{
    displayName: '{lakehouse_name}'
    description: '{lakehouse_desc}'
  }}
}}
'''
        
        # Add warehouses
        warehouses = workspace_config.get("artifacts", {}).get("warehouses", [])
        for warehouse in warehouses:
            warehouse_name = warehouse["name"]
            warehouse_desc = warehouse.get("description", "")
            
            bicep += f'''
resource warehouse_{warehouse_name} 'Microsoft.Fabric/workspaces/warehouses@2023-11-01' = {{
  parent: workspace
  name: '{warehouse_name}'
  properties: {{
    displayName: '{warehouse_name}'
    description: '{warehouse_desc}'
  }}
}}
'''
        
        bicep += f'''
output workspaceId string = workspace.id
output workspaceName string = workspace.name
'''
        
        return bicep
    
    @staticmethod
    def generate_complete_bicep(environment: str, config: Dict) -> str:
        """Generate complete Bicep deployment."""
        env_config = config["environments"][environment]
        
        # Main template
        bicep = f'''// Insurance Fabric Infrastructure - {environment.upper()}
targetScope = 'subscription'

param environment string = '{environment}'
param location string = 'eastus'
param resourceGroupName string = 'insurance-fabric-rg'

// Create resource group
resource rg 'Microsoft.Resources/resourceGroups@2021-04-01' = {{
  name: resourceGroupName
  location: location
  tags: {{
    Project: 'Insurance Accelerator'
    Environment: environment
    ManagedBy: 'Bicep'
  }}
}}

// Deploy capacity
module capacity './capacity.bicep' = {{
  scope: rg
  name: 'fabric-capacity-deployment'
  params: {{
    environment: environment
    location: location
    capacitySku: '{env_config.get("capacity_sku", "F64")}'
  }}
}}
'''
        
        # Deploy workspaces
        for i, workspace_config in enumerate(env_config["workspaces"]):
            workspace_name = workspace_config["name"]
            
            bicep += f'''
// Deploy workspace: {workspace_name}
module workspace_{i} './workspace_{workspace_name.replace("-", "_")}.bicep' = {{
  scope: rg
  name: 'workspace-{workspace_name}-deployment'
  params: {{
    capacityId: capacity.outputs.capacityId
  }}
  dependsOn: [
    capacity
  ]
}}
'''
        
        return bicep

print("\n✅ BicepGenerator class loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: Fabric CLI Deployment Scripts                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

class FabricCLIGenerator:
    """
    Generate Fabric CLI deployment scripts.
    Uses Fabric REST API via Azure CLI.
    """
    
    @staticmethod
    def generate_deployment_script(environment: str, config: Dict) -> str:
        """Generate complete CLI deployment script."""
        
        env_config = config["environments"][environment]
        
        script = f'''#!/bin/bash
# Fabric Infrastructure Deployment - {environment.upper()}
# Generated: {datetime.now().isoformat()}

set -e  # Exit on error

echo "=========================================="
echo "Fabric Infrastructure Deployment"
echo "Environment: {environment.upper()}"
echo "=========================================="

# Configuration
ENVIRONMENT="{environment}"
CAPACITY_SKU="{env_config.get('capacity_sku', 'F64')}"
REGION="{env_config.get('region', 'eastus')}"

# Authenticate with Azure
echo "\\n[1/5] Authenticating with Azure..."
az login --use-device-code

# Get access token for Fabric API
echo "\\n[2/5] Getting Fabric API access token..."
ACCESS_TOKEN=$(az account get-access-token --resource https://api.fabric.microsoft.com --query accessToken -o tsv)

# Fabric API base URL
FABRIC_API="https://api.fabric.microsoft.com/v1"

'''
        
        # Create workspaces
        for i, workspace_config in enumerate(env_config["workspaces"]):
            workspace_name = workspace_config["name"]
            description = workspace_config.get("description", "")
            
            script += f'''
echo "\\n[3/{len(env_config['workspaces'])}] Creating workspace: {workspace_name}..."

# Create workspace
WORKSPACE_RESPONSE=$(curl -X POST "$FABRIC_API/workspaces" \\
  -H "Authorization: Bearer $ACCESS_TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "displayName": "{workspace_name}",
    "description": "{description}",
    "capacityId": "{env_config.get('capacity_id', '')}"
  }}')

WORKSPACE_ID=$(echo $WORKSPACE_RESPONSE | jq -r '.id')
echo "   ✅ Workspace created: $WORKSPACE_ID"

'''
            
            # Create lakehouses
            lakehouses = workspace_config.get("artifacts", {}).get("lakehouses", [])
            for lakehouse in lakehouses:
                lakehouse_name = lakehouse["name"]
                lakehouse_desc = lakehouse.get("description", "")
                
                script += f'''
echo "   Creating lakehouse: {lakehouse_name}..."
curl -X POST "$FABRIC_API/workspaces/$WORKSPACE_ID/lakehouses" \\
  -H "Authorization: Bearer $ACCESS_TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "displayName": "{lakehouse_name}",
    "description": "{lakehouse_desc}"
  }}'
echo "   ✅ Lakehouse created: {lakehouse_name}"
'''
            
            # Create warehouses
            warehouses = workspace_config.get("artifacts", {}).get("warehouses", [])
            for warehouse in warehouses:
                warehouse_name = warehouse["name"]
                warehouse_desc = warehouse.get("description", "")
                
                script += f'''
echo "   Creating warehouse: {warehouse_name}..."
curl -X POST "$FABRIC_API/workspaces/$WORKSPACE_ID/warehouses" \\
  -H "Authorization: Bearer $ACCESS_TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "displayName": "{warehouse_name}",
    "description": "{warehouse_desc}"
  }}'
echo "   ✅ Warehouse created: {warehouse_name}"
'''
            
            # Import notebooks
            notebooks = workspace_config.get("artifacts", {}).get("notebooks", [])
            if notebooks:
                script += f'''
echo "   Importing {len(notebooks)} notebooks..."
for notebook in {" ".join(notebooks)}; do
  echo "      Importing $notebook..."
  curl -X POST "$FABRIC_API/workspaces/$WORKSPACE_ID/notebooks/import" \\
    -H "Authorization: Bearer $ACCESS_TOKEN" \\
    -H "Content-Type: application/json" \\
    -d "{{
      \\"displayName\\": \\"${{notebook%.ipynb}}\\",
      \\"definition\\": {{
        \\"format\\": \\"ipynb\\",
        \\"parts\\": [
          {{
            \\"path\\": \\"notebook-content.ipynb\\",
            \\"payload\\": \\"$(base64 -w 0 notebooks/$notebook)\\"
          }}
        ]
      }}
    }}"
done
echo "   ✅ Notebooks imported"
'''
        
        script += '''
echo "\\n=========================================="
echo "✅ Infrastructure deployment complete!"
echo "=========================================="
'''
        
        return script
    
    @staticmethod
    def generate_python_deployment_script(environment: str, config: Dict) -> str:
        """Generate Python deployment script using requests library."""
        
        env_config = config["environments"][environment]
        
        script = f'''#!/usr/bin/env python3
"""
Fabric Infrastructure Deployment - {environment.upper()}
Generated: {datetime.now().isoformat()}
"""

import requests
import json
import subprocess
import base64
from pathlib import Path

# Configuration
ENVIRONMENT = "{environment}"
CAPACITY_ID = "{env_config.get('capacity_id', '')}"

print("="*60)
print("Fabric Infrastructure Deployment")
print(f"Environment: {{ENVIRONMENT.upper()}}")
print("="*60)

# Get access token
print("\\n[1/4] Getting Fabric API access token...")
token_result = subprocess.run(
    ["az", "account", "get-access-token", "--resource", "https://api.fabric.microsoft.com"],
    capture_output=True,
    text=True
)
access_token = json.loads(token_result.stdout)["accessToken"]

# API setup
headers = {{
    "Authorization": f"Bearer {{access_token}}",
    "Content-Type": "application/json"
}}
api_base = "https://api.fabric.microsoft.com/v1"

# Create workspaces
'''
        
        for workspace_config in env_config["workspaces"]:
            workspace_name = workspace_config["name"]
            description = workspace_config.get("description", "")
            
            script += f'''
print("\\n[2/4] Creating workspace: {workspace_name}...")
workspace_payload = {{
    "displayName": "{workspace_name}",
    "description": "{description}",
    "capacityId": CAPACITY_ID
}}

response = requests.post(
    f"{{api_base}}/workspaces",
    headers=headers,
    json=workspace_payload
)
response.raise_for_status()
workspace_id = response.json()["id"]
print(f"   ✅ Workspace created: {{workspace_id}}")
'''
            
            # Create artifacts
            lakehouses = workspace_config.get("artifacts", {}).get("lakehouses", [])
            if lakehouses:
                script += f'''
print("\\n[3/4] Creating lakehouses...")
lakehouses = {json.dumps(lakehouses, indent=4)}

for lakehouse in lakehouses:
    print(f"   Creating lakehouse: {{lakehouse['name']}}...")
    response = requests.post(
        f"{{api_base}}/workspaces/{{workspace_id}}/lakehouses",
        headers=headers,
        json=lakehouse
    )
    response.raise_for_status()
    print(f"   ✅ Lakehouse created: {{lakehouse['name']}}")
'''
        
        script += '''
print("\\n" + "="*60)
print("✅ Infrastructure deployment complete!")
print("="*60)
'''
        
        return script

print("\n✅ FabricCLIGenerator class loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: Generate All Infrastructure-as-Code Files               ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os

print("\n" + "=" * 80)
print("📝 GENERATING INFRASTRUCTURE-AS-CODE FILES")
print("=" * 80)

# Output directory
iac_dir = "/tmp/infrastructure"
os.makedirs(iac_dir, exist_ok=True)
os.makedirs(f"{iac_dir}/terraform", exist_ok=True)
os.makedirs(f"{iac_dir}/bicep", exist_ok=True)
os.makedirs(f"{iac_dir}/cli", exist_ok=True)

# Generate for each environment
environments = ["dev", "test", "prod"]

for env in environments:
    print(f"\n{'='*60}")
    print(f"Environment: {env.upper()}")
    print(f"{'='*60}")
    
    # === Terraform ===
    print(f"\n📦 Generating Terraform for {env}...")
    
    terraform_main = TerraformGenerator.generate_complete_terraform(env, infrastructure_config_schema)
    
    with open(f"{iac_dir}/terraform/{env}_main.tf", "w") as f:
        f.write(terraform_main)
    
    print(f"   ✅ Generated: terraform/{env}_main.tf")
    
    # === Bicep ===
    print(f"\n📦 Generating Bicep for {env}...")
    
    # Main template
    bicep_main = BicepGenerator.generate_complete_bicep(env, infrastructure_config_schema)
    
    with open(f"{iac_dir}/bicep/{env}_main.bicep", "w") as f:
        f.write(bicep_main)
    
    # Capacity module
    env_config = infrastructure_config_schema["environments"][env]
    bicep_capacity = BicepGenerator.generate_capacity_bicep(env_config)
    
    with open(f"{iac_dir}/bicep/capacity.bicep", "w") as f:
        f.write(bicep_capacity)
    
    # Workspace modules
    for workspace_config in env_config["workspaces"]:
        workspace_name = workspace_config["name"]
        bicep_workspace = BicepGenerator.generate_workspace_bicep(workspace_config)
        
        with open(f"{iac_dir}/bicep/workspace_{workspace_name.replace('-', '_')}.bicep", "w") as f:
            f.write(bicep_workspace)
    
    # Parameters
    bicep_params = BicepGenerator.generate_parameters(env)
    
    with open(f"{iac_dir}/bicep/{env}_parameters.json", "w") as f:
        f.write(bicep_params)
    
    print(f"   ✅ Generated: bicep/{env}_main.bicep")
    print(f"   ✅ Generated: bicep/capacity.bicep")
    print(f"   ✅ Generated: bicep/{env}_parameters.json")
    
    # === Fabric CLI ===
    print(f"\n📦 Generating Fabric CLI for {env}...")
    
    # Bash script
    cli_bash = FabricCLIGenerator.generate_deployment_script(env, infrastructure_config_schema)
    
    with open(f"{iac_dir}/cli/deploy_{env}.sh", "w") as f:
        f.write(cli_bash)
    
    # Make executable
    os.chmod(f"{iac_dir}/cli/deploy_{env}.sh", 0o755)
    
    # Python script
    cli_python = FabricCLIGenerator.generate_python_deployment_script(env, infrastructure_config_schema)
    
    with open(f"{iac_dir}/cli/deploy_{env}.py", "w") as f:
        f.write(cli_python)
    
    os.chmod(f"{iac_dir}/cli/deploy_{env}.py", 0o755)
    
    print(f"   ✅ Generated: cli/deploy_{env}.sh")
    print(f"   ✅ Generated: cli/deploy_{env}.py")

print("\n" + "=" * 80)
print("✅ INFRASTRUCTURE-AS-CODE GENERATION COMPLETE")
print("=" * 80)

print(f"\nGenerated files in: {iac_dir}")
print(f"\n📁 File Structure:")
print(f"   {iac_dir}/")
print(f"   ├── terraform/")
print(f"   │   ├── dev_main.tf")
print(f"   │   ├── test_main.tf")
print(f"   │   └── prod_main.tf")
print(f"   ├── bicep/")
print(f"   │   ├── dev_main.bicep")
print(f"   │   ├── test_main.bicep")
print(f"   │   ├── prod_main.bicep")
print(f"   │   ├── capacity.bicep")
print(f"   │   ├── workspace_*.bicep")
print(f"   │   └── *_parameters.json")
print(f"   └── cli/")
print(f"       ├── deploy_dev.sh")
print(f"       ├── deploy_dev.py")
print(f"       ├── deploy_test.sh")
print(f"       ├── deploy_test.py")
print(f"       ├── deploy_prod.sh")
print(f"       └── deploy_prod.py")

print("\n" + "=" * 80)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6: Deployment Orchestrator                                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

class InfrastructureDeployer:
    """
    Orchestrate infrastructure deployment across environments.
    Supports Terraform, Bicep, and CLI deployments.
    """
    
    def __init__(self, environment: str, method: str = "terraform"):
        """
        Initialize deployer.
        
        Args:
            environment: dev, test, prod
            method: terraform, bicep, cli
        """
        self.environment = environment
        self.method = method
    
    def deploy_with_terraform(self) -> Dict:
        """Deploy using Terraform."""
        print(f"\n🚀 Deploying {self.environment.upper()} with Terraform")
        print("=" * 70)
        
        # Commands to run
        commands = [
            ("terraform init", "Initialize Terraform"),
            ("terraform validate", "Validate configuration"),
            ("terraform plan -out=tfplan", "Generate execution plan"),
            ("terraform apply tfplan", "Apply infrastructure changes")
        ]
        
        print("\n📋 Deployment Steps:")
        for cmd, desc in commands:
            print(f"   {desc}:")
            print(f"      $ {cmd}")
        
        print("\n⚠️  This is a dry-run. To execute:")
        print(f"   1. cd {iac_dir}/terraform")
        print(f"   2. Run the commands above")
        
        return {
            "status": "prepared",
            "environment": self.environment,
            "method": "terraform",
            "next_steps": commands
        }
    
    def deploy_with_bicep(self) -> Dict:
        """Deploy using Bicep."""
        print(f"\n🚀 Deploying {self.environment.upper()} with Bicep")
        print("=" * 70)
        
        # Commands to run
        commands = [
            ("az login", "Authenticate with Azure"),
            (f"az deployment sub create --location eastus --template-file {self.environment}_main.bicep --parameters {self.environment}_parameters.json", 
             "Deploy Bicep template")
        ]
        
        print("\n📋 Deployment Steps:")
        for cmd, desc in commands:
            print(f"   {desc}:")
            print(f"      $ {cmd}")
        
        print("\n⚠️  This is a dry-run. To execute:")
        print(f"   1. cd {iac_dir}/bicep")
        print(f"   2. Run the commands above")
        
        return {
            "status": "prepared",
            "environment": self.environment,
            "method": "bicep",
            "next_steps": commands
        }
    
    def deploy_with_cli(self) -> Dict:
        """Deploy using Fabric CLI."""
        print(f"\n🚀 Deploying {self.environment.upper()} with Fabric CLI")
        print("=" * 70)
        
        script_path = f"{iac_dir}/cli/deploy_{self.environment}.sh"
        
        print(f"\n📋 Deployment Script: {script_path}")
        print("\nTo execute:")
        print(f"   $ chmod +x {script_path}")
        print(f"   $ {script_path}")
        
        print("\nOr use Python script:")
        print(f"   $ python {iac_dir}/cli/deploy_{self.environment}.py")
        
        return {
            "status": "prepared",
            "environment": self.environment,
            "method": "cli",
            "script_path": script_path
        }
    
    def deploy(self) -> Dict:
        """Execute deployment based on method."""
        
        if self.method == "terraform":
            return self.deploy_with_terraform()
        elif self.method == "bicep":
            return self.deploy_with_bicep()
        elif self.method == "cli":
            return self.deploy_with_cli()
        else:
            raise ValueError(f"Unknown deployment method: {self.method}")

print("\n✅ InfrastructureDeployer class loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 7: Execute Sample Deployment                               ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("🎯 SAMPLE DEPLOYMENT EXECUTION")
print("=" * 80)

# === Deploy Development Environment ===
deployer_dev = InfrastructureDeployer("dev", "terraform")
result_dev = deployer_dev.deploy()

print(f"\n✅ Deployment prepared")
print(f"   Environment: {result_dev['environment']}")
print(f"   Method: {result_dev['method']}")
print(f"   Status: {result_dev['status']}")

# === Deployment Status Tracking ===
print("\n" + "=" * 80)
print("📊 DEPLOYMENT STATUS TRACKING")
print("=" * 80)

# Track deployments in Delta table
deployment_log = spark.createDataFrame([{
    "deployment_id": f"DEP_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    "environment": "dev",
    "method": "terraform",
    "status": "prepared",
    "timestamp": datetime.now(),
    "deployed_by": notebookutils.runtime.context.get("userEmail", "system")
}])

deployment_log.write.format("delta") \
    .mode("append") \
    .saveAsTable("metadata.infrastructure_deployments")

print("✅ Deployment logged to metadata.infrastructure_deployments")

# === Show Deployment History ===
deployment_history = spark.sql("""
    SELECT 
        deployment_id,
        environment,
        method,
        status,
        timestamp,
        deployed_by
    FROM metadata.infrastructure_deployments
    ORDER BY timestamp DESC
    LIMIT 10
""")

print("\n📋 Recent Deployments:")
display(deployment_history)

print("\n" + "=" * 80)

## 🚀 Deployment Guide

### Method 1: Terraform (Recommended for Enterprise)

**Prerequisites:**
- Terraform >= 1.6
- Azure CLI authenticated
- Fabric capacity pre-created

**Deploy Development:**
```bash
cd /tmp/infrastructure/terraform
terraform init
terraform plan -var-file="dev.tfvars"
terraform apply -var-file="dev.tfvars"
```

**Deploy Production:**
```bash
terraform plan -var-file="prod.tfvars"
terraform apply -var-file="prod.tfvars"
```

**Destroy (when needed):**
```bash
terraform destroy -var-file="dev.tfvars"
```

---

### Method 2: Bicep (Azure-Native)

**Prerequisites:**
- Azure CLI >= 2.50.0
- Bicep CLI installed
- Azure subscription with permissions

**Deploy Development:**
```bash
cd /tmp/infrastructure/bicep
az deployment sub create \
  --location eastus \
  --template-file dev_main.bicep \
  --parameters dev_parameters.json
```

**Deploy Production:**
```bash
az deployment sub create \
  --location eastus2 \
  --template-file prod_main.bicep \
  --parameters prod_parameters.json
```

---

### Method 3: Fabric CLI (Quick Deployment)

**Prerequisites:**
- Azure CLI authenticated
- jq installed (for bash) or requests package (for Python)

**Deploy with Bash:**
```bash
cd /tmp/infrastructure/cli
./deploy_dev.sh
```

**Deploy with Python:**
```python
cd /tmp/infrastructure/cli
python deploy_dev.py
```

---

## 🔧 Configuration Management

### Environment Variables (Terraform)

Create `dev.tfvars`:
```hcl
environment          = "dev"
location            = "eastus"
capacity_sku        = "F2"
resource_group_name = "insurance-fabric-dev-rg"

capacity_admins = [
  "dev-admin@company.com"
]

tags = {
  Project     = "Insurance Accelerator"
  Environment = "dev"
  ManagedBy   = "Terraform"
  CostCenter  = "DataPlatform"
}
```

Create `prod.tfvars`:
```hcl
environment          = "prod"
location            = "eastus2"
capacity_sku        = "F64"
resource_group_name = "insurance-fabric-prod-rg"

capacity_admins = [
  "prod-admin@company.com",
  "platform-team@company.com"
]

tags = {
  Project     = "Insurance Accelerator"
  Environment = "prod"
  ManagedBy   = "Terraform"
  CostCenter  = "DataPlatform"
  Compliance  = "SOC2"
}
```

---

## 📊 Monitoring Deployment Status

Query deployment history:
```sql
SELECT 
    environment,
    method,
    status,
    COUNT(*) as deployment_count,
    MAX(timestamp) as last_deployment
FROM metadata.infrastructure_deployments
GROUP BY environment, method, status
ORDER BY last_deployment DESC
```

---

## 🎯 Best Practices

### 1. State Management (Terraform)

Use remote state for team collaboration:
```hcl
terraform {
  backend "azurerm" {
    resource_group_name  = "insurance-tfstate-rg"
    storage_account_name = "insurancetfstate"
    container_name       = "tfstate"
    key                  = "fabric-prod.terraform.tfstate"
  }
}
```

### 2. Environment Separation

- Separate Azure subscriptions for Prod
- Different capacity SKUs per environment
- Environment-specific RBAC

### 3. CI/CD Integration

**GitHub Actions:**
```yaml
name: Deploy to Development

on:
  push:
    branches: [develop]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      
      - name: Setup Terraform
        uses: hashicorp/setup-terraform@v2
      
      - name: Terraform Init
        run: terraform init
        working-directory: ./infrastructure/terraform
      
      - name: Terraform Apply
        run: terraform apply -auto-approve -var-file="dev.tfvars"
        working-directory: ./infrastructure/terraform
```

### 4. Drift Detection

Run daily drift detection:
```bash
terraform plan -detailed-exitcode -var-file="prod.tfvars"
# Exit code 2 = drift detected
```

---

## 🔒 Security Considerations

- Store credentials in Azure Key Vault
- Use Managed Identities for authentication
- Enable audit logging for all deployments
- Implement approval gates for Production
- Encrypt Terraform state files
- Use role-based access for deployments

---

## ✅ Summary

This notebook provides:

✅ **Terraform** — Enterprise-grade IaC with state management  
✅ **Bicep** — Azure-native declarative deployment  
✅ **Fabric CLI** — Quick scripted deployment  
✅ **Multi-environment** — Dev, Test, Prod configurations  
✅ **Complete artifact support** — Lakehouses, warehouses, notebooks, KQL, mirroring  
✅ **CI/CD ready** — GitHub Actions, Azure DevOps integration  
✅ **Deployment tracking** — Delta table audit logging  
✅ **Modular** — Reusable components  

**Production-ready enterprise infrastructure deployment!** 🏗️